# Create a dependency graph of items


Build a dependency graph of items, detect broken nodes, and export a sorted report of item connection degrees.


## 1. Import libraries and connect to GIS


In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from arcgis.apps.itemgraph import create_dependency_graph
from arcgis.gis import GIS

In [ ]:
gis = GIS("home")

## 2. Retrieve items from content


In [ ]:
my_items = gis.content.search(query=f"owner:{gis.users.me.username}", max_items=-1)
print(f"Total number of items: {len(my_items)}")

## 3. Create dependency graph


In [ ]:
graph = create_dependency_graph(gis, my_items)
all_nodes = graph.all_items()

In [ ]:
pos = nx.spring_layout(graph)  # positions for all nodes
nx.draw(
    graph,
    pos,
    with_labels=True,
    node_size=500,
    node_color="skyblue",
    font_size=6,
    font_color="green",
)
nx.draw_networkx_labels(graph, pos, labels=nx.get_node_attributes(graph, "label"))
plt.show()

## 4. Find broken dependencies


Nodes (items) without any attached items indicate that they are either inaccessible within this organization or do not exist.


In [ ]:
for node in all_nodes:
    if node.item == None:
        print(node.id)

## 5. Sort items by degree


The following methods help compile a list of items that a `node` (item) either requires to exist or that require the item to exist:

- `node.contains()` - Items that this node directly contains
- `node.contained_by()` - Items that directly contain this item
- `node.requires()` - Deep dependencies this item requires to exist
- `node.required_by()` - Deep reverse dependencies that require this item to exist

The following code sorts items by degree based on the items they directly contain. To sort items by a different relationship degree, replace `node.contains()` with the appropriate method.


In [ ]:
nodes_weighted = [(node, len(node.contains())) for node in all_nodes]

In [ ]:
rows = []

for node, degree in sorted(nodes_weighted, key=lambda x: x[1], reverse=True):
    item = getattr(node, "item", None)
    rows.append(
        {
            "title": getattr(item, "title", None),
            "item_id": node.id,
            "item_type": getattr(item, "type", None),
            "degree": degree,
        }
    )

dependency_df = pd.DataFrame.from_records(
    rows, columns=["title", "item_id", "item_type", "degree"]
)

In [ ]:
print("Preview (top 10):")
print(dependency_df.head(10))

## 6. Save report to CSV


In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d")
file_path = rf"/arcgis/home/item_graph_dependency_{timestamp}.csv"
dependency_df.to_csv(file_path)